In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, auc, average_precision_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

from config.paths import (
    X_TRAIN, X_TEST, Y_TRAIN, Y_TEST,
    RANDOM_FOREST_METRICS, RANDOM_FOREST_MODEL, RANDOM_FOREST_CHARTS
)

RANDOM_FOREST_CHARTS.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(X_TRAIN)
X_test = pd.read_csv(X_TEST)
y_train = pd.read_csv(Y_TRAIN).squeeze()
y_test = pd.read_csv(Y_TEST).squeeze()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 7), (2000, 7), (8000,), (2000,))

In [4]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

scoring = ['precision', 'recall', 'f1', 'roc_auc', 'average_precision']
cv_results = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring)

cv_summary = pd.DataFrame({
    metric: [cv_results[f'test_{metric}'].mean(), cv_results[f'test_{metric}'].std()]
    for metric in scoring
}, index=['mean', 'std']).T

print(cv_summary)

                       mean       std
precision          0.732806  0.076609
recall             0.697354  0.059476
f1                 0.713742  0.062747
roc_auc            0.968402  0.017293
average_precision  0.774811  0.052292
